# 02 — L1 NiO single-orbital SIAM + IBM hardware run

**Goal:** VQE on a 4-spin-orbital Anderson impurity reduction of NiO,
with parameters from EDRIXS example_3 (Haverkort PRB 85, 165113).
Execute on (a) noiseless simulator, (b) FakeMarrakesh noisy sim,
(c) real IBM Quantum hardware at 3 resilience levels.

**Honesty note:** this is *not* a literal d⁸ NiO calculation. It's a
half-filled 2-mode toy reduction that inherits its parameters from
the EDRIXS example_3 NiO Anderson impurity model.

**Validation layers** (spec §6):
1. Energy match against ED (|ΔE| < 1 mHa noiseless)
2. State overlap ≥ 0.99 noiseless
3. Ansatz expressivity ≥ 0.99
4. Multi-start spread (N=8) best-to-median < 1 mHa
5. Observable agreement (n_d, n_p, S²_d, double_occ_d) within 1%
6. Hardware resilience-tier monotonicity (informational)

Layers 1-5 are HARD GATES on noiseless. Hardware submission is
skipped if any of those fail.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import siam_vqe as svqe
from siam_vqe import (
    check_ansatz_expressivity, check_energy_match, check_multistart_spread,
    check_observable_agreement_multi, check_resilience_guardrail,
    check_state_overlap, compare_energies, compute_l1_levels, exact_diag,
    make_noisy_estimator, nio_l1_anderson, observables_l1, run_vqe,
    run_vqe_multistart, to_qubit_op, uccsd_ansatz,
)
from siam_vqe.hardware import (
    make_runtime_estimator, pick_backend, transpile_for_backend,
)
print(f'siam_vqe {svqe.__version__}')

## 1. L1 parameters (from EDRIXS example_3 + CT_imp_bath)

U_dd, V_eg, Δ, 10Dq, ten_dq_bath: verbatim from example_3.
(ε_d, ε_p) computed via closed-form CT_imp_bath port + eg crystal-field shift.


In [2]:
U = 7.3       # U_dd
V = 2.06      # V_eg
eps_d, eps_p = compute_l1_levels()
print(f'U = {U} eV')
print(f'V = {V} eV')
print(f'eps_d = {eps_d:.6f} eV')
print(f'eps_p = {eps_p:.6f} eV')
print(f'eps_d - eps_p = {eps_d - eps_p:.6f} eV')


U = 7.3 eV
V = 2.06 eV
eps_d = -40.852889 eV
eps_p = 13.375111 eV
eps_d - eps_p = -54.228000 eV


## 2. Build H + observables + ED reference


In [3]:
fop = nio_l1_anderson(U=U, V=V, eps_d=eps_d, eps_p=eps_p)
num_particles = (1, 1)
pop = to_qubit_op(fop, scheme='parity_tapered', num_particles=num_particles)
obs = observables_l1()
print(f'qubits (parity_tapered): {pop.num_qubits}')
print(f'Pauli terms: {len(pop)}')

ed = exact_diag(pop, k=4)
print(f'ED ground state E0 = {ed.energies[0]:.8f} eV')
print(f'lowest 4: {np.round(ed.energies, 6)}')


qubits (parity_tapered): 2
Pauli terms: 6
ED ground state E0 = -74.58626153 eV
lowest 4: [-74.586262 -27.477778 -27.453595  26.906523]


## 3. Noiseless VQE (UCCSD + SLSQP, N=8 multi-start)


In [4]:
circuit, x0 = uccsd_ansatz(
    num_spatial_orbitals=2,
    num_particles=num_particles,
    mapper_scheme='parity_tapered',
)
print(f'ansatz: UCCSD, {circuit.num_parameters} parameters')

multistart = run_vqe_multistart(
    pop, circuit, n_starts=8, optimizer='SLSQP', maxiter=200, seed=20260524,
)
best = multistart.best
print(f'best E = {best.energy:.8f} eV  ({best.optimizer_name})')
print(f'spread (best-to-median) = {multistart.spread_best_to_median:.2e}')


ansatz: UCCSD, 3 parameters


best E = -74.58626153 eV  (SLSQP)
spread (best-to-median) = 1.44e-08


## 4. Validation layers 1-5 (hard gate)

All five must pass before submitting any hardware job.


In [ ]:
# Layer 1: energy match
em = check_energy_match(best.energy, ed.energies[0], abs_tol=1e-3)
print(f'Layer 1 (energy match): passed={em.passed} |ΔE|={em.delta_e:.3e}')

# Layer 2: state overlap
ov = check_state_overlap(best, pop, ed.vectors[:, 0])
print(f'Layer 2 (state overlap): passed={ov.passed} overlap²={ov.overlap_sq:.6f}')

# Layer 3: ansatz expressivity
exp = check_ansatz_expressivity(circuit, ed.vectors[:, 0], n_starts=8, seed=20260524)
print(f'Layer 3 (expressivity): passed={exp.passed} max-overlap={exp.max_overlap:.6f}')

# Layer 4: multi-start spread
ms = check_multistart_spread(multistart, tol_mha=1.0)
print(
    f'Layer 4 (multi-start spread): passed={ms.passed} '
    f'best-to-median={ms.spread_best_to_median:.3e}'
)

# Layer 5: observable agreement
ob = check_observable_agreement_multi(best, pop, ed.vectors[:, 0], obs, num_particles=(1, 1), rel_tol=0.01)
print(f'Layer 5 (observables): passed={ob.passed} max_rel_err={ob.max_rel_error:.3e}')
for k, v in ob.values.items():
    print(f'  {k}: VQE={v.vqe:.6f}  ED={v.ed:.6f}  rel_err={v.rel_error:.3e}')

all_passed = em.passed and ov.passed and exp.passed and ms.passed and ob.passed
assert all_passed, 'Noiseless validation failed; aborting before hardware submit.'
print('\nAll five noiseless layers PASSED — clear to proceed to FakeMarrakesh + hardware.')

## 5. FakeMarrakesh noisy simulator at x*


In [6]:
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit import transpile

bound_circuit = circuit.assign_parameters(best.params)
fake = FakeMarrakesh()
isa_circ_fake = transpile(bound_circuit, backend=fake, optimization_level=3)
isa_pop_fake = pop.apply_layout(isa_circ_fake.layout)
noisy_est = make_noisy_estimator(fake, shots=8192, seed=20260524)

job_fake = noisy_est.run([(isa_circ_fake, isa_pop_fake)])
res_fake = job_fake.result()
E_fake = float(res_fake[0].data.evs)
E_fake_std = float(res_fake[0].data.stds) if hasattr(res_fake[0].data, 'stds') else 0.0
print(f'FakeMarrakesh: E = {E_fake:.6f} ± {E_fake_std:.6f} eV')
print(f'gap to ED:    |ΔE| = {abs(E_fake - ed.energies[0]):.6f} eV')


FakeMarrakesh: E = -73.971928 ± 0.153373 eV
gap to ED:    |ΔE| = 0.614333 eV


## 6. Partial result snapshot (pre-hardware)


In [7]:
partial = {
    'phase': 2,
    'level': 'L1',
    'parameters': {'U': U, 'V': V, 'eps_d': eps_d, 'eps_p': eps_p},
    'ed_energy': float(ed.energies[0]),
    'noiseless_energy': float(best.energy),
    'noiseless_params': best.params.tolist(),
    'multistart_spread': float(multistart.spread_best_to_median),
    'fake_marrakesh_energy': E_fake,
    'fake_marrakesh_std': E_fake_std,
    'validation_layers': {
        '1_energy_match': bool(em.passed),
        '2_state_overlap': bool(ov.passed),
        '3_expressivity': bool(exp.passed),
        '4_multistart_spread': bool(ms.passed),
        '5_observable_agreement': bool(ob.passed),
    },
}
partial_path = Path('02_L1_partial_result.json')
partial_path.write_text(json.dumps(partial, indent=2))
print(f'wrote {partial_path}')


wrote 02_L1_partial_result.json


---

## 7. Hardware execution

**Queue expectation:** open-plan jobs can wait 12-48 h end-to-end.
Job IDs are captured to `02_L1_job_ids.json` immediately after
submission so this notebook can be killed and resumed.

Three EstimatorV2 jobs at resilience_level = {0, 1, 2}.
Each evaluates ⟨H⟩ at x* with 8192 shots.


In [8]:
# Set SUBMIT_HARDWARE = True to actually submit. Default False to prevent
# accidental open-plan-budget burn when re-executing the notebook.
SUBMIT_HARDWARE = False

if SUBMIT_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = pick_backend(service, min_qubits=2)
    print(f'backend: {backend.name} (pending_jobs={backend.status().pending_jobs})')

    isa_circ_hw, isa_pop_hw = transpile_for_backend(bound_circuit, pop, backend)
    print(f'ISA circuit: {isa_circ_hw.num_qubits} qubits, depth {isa_circ_hw.depth()}')

    job_ids = {}
    for rl in [0, 1, 2]:
        est = make_runtime_estimator(backend, resilience_level=rl, default_shots=8192)
        job = est.run([(isa_circ_hw, isa_pop_hw)])
        job_ids[f'hw_L{rl}'] = job.job_id()
        print(f'  resilience_level={rl}: job_id={job.job_id()}')

    submission = {
        'backend': backend.name,
        'submitted_at': time.time(),
        'job_ids': job_ids,
        'shots': 8192,
    }
    Path('02_L1_job_ids.json').write_text(json.dumps(submission, indent=2))
    print('wrote 02_L1_job_ids.json')
else:
    print('SUBMIT_HARDWARE = False — set to True and re-run this cell to submit.')


SUBMIT_HARDWARE = False — set to True and re-run this cell to submit.


## 8. Retrieve hardware results (blocking)


In [9]:
from qiskit_ibm_runtime import QiskitRuntimeService

job_ids_path = Path('02_L1_job_ids.json')
if not job_ids_path.exists():
    raise FileNotFoundError(
        '02_L1_job_ids.json not found — set SUBMIT_HARDWARE=True in the '
        'previous cell and re-execute it first.'
    )
submission = json.loads(job_ids_path.read_text())
service = QiskitRuntimeService()

hw_results: dict[str, dict[str, float]] = {}
for label, jid in submission['job_ids'].items():
    job = service.job(jid)
    print(f'  {label} ({jid}): status = {job.status()}')
    res = job.result()
    pub = res[0]
    ev = float(pub.data.evs)
    std = float(pub.data.stds) if hasattr(pub.data, 'stds') else 0.0
    hw_results[label] = {'energy': ev, 'std': std}
    print(f'    E = {ev:.6f} ± {std:.6f} eV')


FileNotFoundError: 02_L1_job_ids.json not found — set SUBMIT_HARDWARE=True in the previous cell and re-execute it first.

## 9. Layer 6 guardrail + comparison plot + final JSON


In [10]:
# Layer 6 guardrail (informational; pass-or-flag, never aborts)
guardrail = check_resilience_guardrail(
    e_ed=float(ed.energies[0]),
    e_hw_l0=hw_results['hw_L0']['energy'],
    e_hw_l1=hw_results['hw_L1']['energy'],
    e_hw_l2=hw_results['hw_L2']['energy'],
)
print(f'Layer 6 (resilience guardrail): passed={guardrail.passed}')
print(f'  gap_L0 = {guardrail.gap_l0:.6f} eV')
print(f'  gap_L1 = {guardrail.gap_l1:.6f} eV')
print(f'  gap_L2 = {guardrail.gap_l2:.6f} eV')
print(f'  notes: {guardrail.notes}')

# Comparison figure
all_energies = {
    'ED': float(ed.energies[0]),
    'noiseless': float(best.energy),
    'FakeMarrakesh': E_fake,
    'hw_L0': hw_results['hw_L0']['energy'],
    'hw_L1': hw_results['hw_L1']['energy'],
    'hw_L2': hw_results['hw_L2']['energy'],
}
uncertainties = {
    'FakeMarrakesh': E_fake_std,
    'hw_L0': hw_results['hw_L0']['std'],
    'hw_L1': hw_results['hw_L1']['std'],
    'hw_L2': hw_results['hw_L2']['std'],
}
fig = compare_energies(
    all_energies, ed_energy=float(ed.energies[0]), uncertainties=uncertainties,
    title=(
        'L1 NiO SIAM — noiseless / FakeMarrakesh / hardware '
        '(3 mitigation tiers) vs. ED'
    ),
)
fig_dir = Path('../figures'); fig_dir.mkdir(exist_ok=True)
fig.savefig(fig_dir / '02_L1_energy_comparison.png', dpi=150)
fig.savefig(fig_dir / '02_L1_energy_comparison.pdf')
plt.show()

# Full result JSON
final = dict(partial)
final['backend'] = submission['backend']
final['hw_results'] = hw_results
final['validation_layers']['6_resilience_guardrail'] = bool(guardrail.passed)
final['guardrail'] = {
    'passed': bool(guardrail.passed),
    'gap_l0': guardrail.gap_l0,
    'gap_l1': guardrail.gap_l1,
    'gap_l2': guardrail.gap_l2,
    'notes': guardrail.notes,
}
Path('02_L1_vqe_result.json').write_text(json.dumps(final, indent=2))
print('wrote 02_L1_vqe_result.json')


NameError: name 'hw_results' is not defined